In [1]:

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName('AzureDE-InterviewPrep')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)
spark

In [2]:

spark.createDataFrame(
    [
        ('e1', 'Sales', 70000),
        ('e2', 'Sales', 80000),
        ('e3', 'Sales', 80000),
        ('e4', 'Eng', 90000),
        ('e5', 'Eng', 95000),
        ('e6', 'Eng', 85000),
    ],
    ['emp_id', 'dept', 'salary'],
).createOrReplaceTempView('emp')

# A1. Employees above dept average.

In [3]:
result_a1 = spark.sql("""
    SELECT
        emp_id,
        dept,
        salary,
        AVG(salary) OVER (
            PARTITION BY dept
        ) AS dept_avg_salary
    FROM emp
""")

result_a1.show()

+------+-----+------+-----------------+
|emp_id| dept|salary|  dept_avg_salary|
+------+-----+------+-----------------+
|    e4|  Eng| 90000|          90000.0|
|    e5|  Eng| 95000|          90000.0|
|    e6|  Eng| 85000|          90000.0|
|    e1|Sales| 70000|76666.66666666667|
|    e2|Sales| 80000|76666.66666666667|
|    e3|Sales| 80000|76666.66666666667|
+------+-----+------+-----------------+



In [4]:
result_a1 = spark.sql("""
    SELECT
        emp_id,
        dept,
        salary,
        dept_avg_salary
    FROM (
        SELECT
            emp_id,
            dept,
            salary,
            AVG(salary) OVER (
                PARTITION BY dept
            ) AS dept_avg_salary
        FROM emp
    ) t
    WHERE salary > dept_avg_salary
    ORDER BY dept, salary DESC
""")

result_a1.show()

+------+-----+------+-----------------+
|emp_id| dept|salary|  dept_avg_salary|
+------+-----+------+-----------------+
|    e5|  Eng| 95000|          90000.0|
|    e2|Sales| 80000|76666.66666666667|
|    e3|Sales| 80000|76666.66666666667|
+------+-----+------+-----------------+



# A2. RANK and DENSE_RANK by Department Salary

In [5]:
result_rank = spark.sql("""
    SELECT
        emp_id,
        dept,
        salary,
        RANK() OVER (
            PARTITION BY dept
            ORDER BY salary DESC
        ) AS salary_rank
    FROM emp
    ORDER BY dept, salary DESC
""")

result_rank.show()

+------+-----+------+-----------+
|emp_id| dept|salary|salary_rank|
+------+-----+------+-----------+
|    e5|  Eng| 95000|          1|
|    e4|  Eng| 90000|          2|
|    e6|  Eng| 85000|          3|
|    e2|Sales| 80000|          1|
|    e3|Sales| 80000|          1|
|    e1|Sales| 70000|          3|
+------+-----+------+-----------+



In [6]:
result_dense_rank = spark.sql("""
    SELECT
        emp_id,
        dept,
        salary,
        DENSE_RANK() OVER (
            PARTITION BY dept
            ORDER BY salary DESC
        ) AS dense_salary_rank
    FROM emp
    ORDER BY dept, salary DESC
""")

result_dense_rank.show()

+------+-----+------+-----------------+
|emp_id| dept|salary|dense_salary_rank|
+------+-----+------+-----------------+
|    e5|  Eng| 95000|                1|
|    e4|  Eng| 90000|                2|
|    e6|  Eng| 85000|                3|
|    e2|Sales| 80000|                1|
|    e3|Sales| 80000|                1|
|    e1|Sales| 70000|                2|
+------+-----+------+-----------------+



# A3. Running total of salary by dept.

In [8]:
result_a3 = spark.sql("""
    SELECT
        emp_id,
        dept,
        salary,
        SUM(salary) OVER (
            PARTITION BY dept
            ORDER BY salary
            ROWS BETWEEN UNBOUNDED PRECEDING
                     AND CURRENT ROW
        ) AS running_salary
    FROM emp
    ORDER BY dept, salary
""")

result_a3.show()

+------+-----+------+--------------+
|emp_id| dept|salary|running_salary|
+------+-----+------+--------------+
|    e6|  Eng| 85000|         85000|
|    e4|  Eng| 90000|        175000|
|    e5|  Eng| 95000|        270000|
|    e1|Sales| 70000|         70000|
|    e2|Sales| 80000|        150000|
|    e3|Sales| 80000|        230000|
+------+-----+------+--------------+



# A4. LAG — Month-over-Month Style Change

In [10]:
monthly_salary = spark.createDataFrame(
    [
        ("e1", "2024-01-01", 70000),
        ("e1", "2024-02-01", 75000),
        ("e1", "2024-03-01", 80000),
        ("e2", "2024-01-01", 80000),
        ("e2", "2024-02-01", 82000),
        ("e2", "2024-03-01", 85000),
    ],
    ["emp_id", "month", "salary"]
)

monthly_salary = monthly_salary.withColumn(
    "month",
    F.to_date("month")
)

monthly_salary.createOrReplaceTempView("monthly_salary")

In [12]:
result_a4 = spark.sql("""
    SELECT
        emp_id,
        month,
        salary,
        LAG(salary) OVER (
            PARTITION BY emp_id
            ORDER BY month
        ) AS previous_salary
    FROM monthly_salary
    ORDER BY emp_id, month
""")

result_a4.show()

+------+----------+------+---------------+
|emp_id|     month|salary|previous_salary|
+------+----------+------+---------------+
|    e1|2024-01-01| 70000|           NULL|
|    e1|2024-02-01| 75000|          70000|
|    e1|2024-03-01| 80000|          75000|
|    e2|2024-01-01| 80000|           NULL|
|    e2|2024-02-01| 82000|          80000|
|    e2|2024-03-01| 85000|          82000|
+------+----------+------+---------------+



In [14]:
result_a4 = spark.sql("""
    SELECT
        emp_id,
        month,
        salary,
        previous_salary,
        salary - previous_salary AS salary_change
    FROM (
        SELECT
            emp_id,
            month,
            salary,
            LAG(salary) OVER (
                PARTITION BY emp_id
                ORDER BY month
            ) AS previous_salary
        FROM monthly_salary
    ) t
    ORDER BY emp_id, month
""")

result_a4.show()

result_a4 = spark.sql("""
    SELECT
        emp_id,
        month,
        salary,
        previous_salary,
        salary - previous_salary AS salary_change,
        ROUND(
            ((salary - previous_salary) / previous_salary) * 100,
            2
        ) AS percentage_change
    FROM (
        SELECT
            emp_id,
            month,
            salary,
            LAG(salary) OVER (
                PARTITION BY emp_id
                ORDER BY month
            ) AS previous_salary
        FROM monthly_salary
    ) t
    ORDER BY emp_id, month
""")

result_a4.show()

+------+----------+------+---------------+-------------+
|emp_id|     month|salary|previous_salary|salary_change|
+------+----------+------+---------------+-------------+
|    e1|2024-01-01| 70000|           NULL|         NULL|
|    e1|2024-02-01| 75000|          70000|         5000|
|    e1|2024-03-01| 80000|          75000|         5000|
|    e2|2024-01-01| 80000|           NULL|         NULL|
|    e2|2024-02-01| 82000|          80000|         2000|
|    e2|2024-03-01| 85000|          82000|         3000|
+------+----------+------+---------------+-------------+

+------+----------+------+---------------+-------------+-----------------+
|emp_id|     month|salary|previous_salary|salary_change|percentage_change|
+------+----------+------+---------------+-------------+-----------------+
|    e1|2024-01-01| 70000|           NULL|         NULL|             NULL|
|    e1|2024-02-01| 75000|          70000|         5000|             7.14|
|    e1|2024-03-01| 80000|          75000|         500

# A5. RANK vs DENSE_RANK vs ROW_NUMBER

In [18]:
result = spark.sql("""
SELECT
    emp_id,
    dept,
    salary,

    RANK() OVER (
        PARTITION BY dept
        ORDER BY salary DESC
    ) AS rank_salary,

    DENSE_RANK() OVER (
        PARTITION BY dept
        ORDER BY salary DESC
    ) AS dense_rank_salary,

    ROW_NUMBER() OVER (
        PARTITION BY dept
        ORDER BY salary DESC, emp_id
    ) AS row_number_salary

FROM emp
ORDER BY dept, salary DESC, emp_id
""")

result.show()

+------+-----+------+-----------+-----------------+-----------------+
|emp_id| dept|salary|rank_salary|dense_rank_salary|row_number_salary|
+------+-----+------+-----------+-----------------+-----------------+
|    e5|  Eng| 95000|          1|                1|                1|
|    e4|  Eng| 90000|          2|                2|                2|
|    e6|  Eng| 85000|          3|                3|                3|
|    e2|Sales| 80000|          1|                1|                1|
|    e3|Sales| 80000|          1|                1|                2|
|    e1|Sales| 70000|          3|                2|                3|
+------+-----+------+-----------+-----------------+-----------------+

